In [11]:
import os
os.chdir("../")

In [12]:
from dataclasses import dataclass 
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir:Path
    base_model_path:Path
    updated_base_model:Path
    params_image_size:list
    params_learning_rate:float
    params_include_top:bool
    params_weights:str
    params_classes:int

In [13]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories



In [19]:
class ConfigurationManager:
 def __init__(
    self,
    config_filepath=CONFIG_FILE_PATH,
    params_filepath=PARAMS_FILE_PATH):
        
    self.config=read_yaml(config_filepath)
    self.params=read_yaml(params_filepath)   
    
    create_directories([self.config.artifacts_root]) 
    

 def get_prepare_base_model_config(self)->PrepareBaseModelConfig:
    config=self.config.prepare_base_model
    
    create_directories([config.root_dir])
    
    prepare_base_model_config=PrepareBaseModelConfig(
        root_dir=Path(config.root_dir),
        base_model_path=Path(config.base_model_path),
        updated_base_model=Path(config.updated_base_model_path),
        params_image_size=self.params.IMAGE_SIZE,
        params_learning_rate=self.params.LEARNING_RATE,
        params_include_top=self.params.INCLUDE_TOP,
        params_weights=self.params.WEIGHTS,
        params_classes=self.params.CLASSES
    )
    
    return prepare_base_model_config

In [20]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf


In [25]:
class PrepareBaseModel:
    def __init__(self,config:PrepareBaseModelConfig):
         self.config=config
         
    def get_base_model(self):
        self.model=tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top)
        
        self.save_model(path=self.config.base_model_path,model=self.model)     
    
    
    @staticmethod
    def save_model(path:Path,model:tf.keras.Model):
           model.save(path)
           
    
    @staticmethod
    def prepare_full_model(model,classes,freeze_all,freeze_till,learning_rate):  
        if freeze_all:
            for layer in model.layers:
                model.trainable=False
        elif(freeze_till is not None)and(freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                model.trainable=False
        
        flatten_in=tf.keras.layers.Flatten()(model.output)
        prediction=tf.keras.layers.Dense(units=classes,
                                         activation='softmax'
                                         )(flatten_in)
        
        full_model=tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )      
        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )   
        
        full_model.summary()
        return full_model           
    
    def update_base_model(self):
        self.full_model=self.prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )
        
        self.save_model(path=self.config.updated_base_model,model=self.full_model)
        
        
    @staticmethod    
    def save_model(path:Path,model:tf.keras.Model):
        model.save(path)

In [26]:
try:
    config=ConfigurationManager()
    prepare_base_model_config=config.get_prepare_base_model_config()
    prepare_base_model=PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e
    

[{'name': 'cnnClassifierLogger', 'msg': 'yaml file:config\\config.yaml loaded successfully', 'args': (), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'c:\\users\\acer nitro v 15\\desktop\\kidney disease classificaton\\src\\cnnClassifier\\utils\\common.py', 'filename': 'common.py', 'module': 'common', 'exc_info': None, 'exc_text': None, 'stack_info': None, 'lineno': 30, 'funcName': 'read_yaml', 'created': 1778915048.6842427, 'msecs': 684.0, 'relativeCreated': 7750535.033226013, 'thread': 380788, 'threadName': 'MainThread', 'processName': 'MainProcess', 'process': 380120, 'message': 'yaml file:config\\config.yaml loaded successfully'}(asctime)s:INFO:common:yaml file:config\config.yaml loaded successfully]
[{'name': 'cnnClassifierLogger', 'msg': 'yaml file:params.yaml loaded successfully', 'args': (), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'c:\\users\\acer nitro v 15\\desktop\\kidney disease classificaton\\src\\cnnClassifier\\utils\\common.py', 'filename': 'common.py', 'module